# Multimodal UAV Detector

This notebook implements the Multimodal UAV Detector. It contains the following functions: 

1. check that the released YOLO datasets match the paper counts;
2. define the multitask YOLOv8n model used by the project;
3. inspect the released checkpoints;
4. run the model evaluation when the experiment package is available;
5. rerun the synthetic fine-tuning stage when needed.


## How to Run It

Open the notebook from the repository root or from `notebooks/`.

The default settings do the light checks first: dataset numbers and checkpoint metadata. To run the full model evaluation or fine-tuning, turn on the switches in the next cell. Those cells need PyTorch, Ultralytics, TorchVision, OpenCV, and the experiment package used during training.


In [ ]:
from __future__ import annotations

import json
import os
import platform
import subprocess
import sys
from collections import Counter
from pathlib import Path

# Leave this off if the environment is already set up.
# Turn it on in Colab or a fresh virtual environment.
RUN_INSTALL_DEPS = False

# Counting every image and label can take a minute on the full dataset.
RUN_DATASET_SCAN = True

# These use the training/evaluation package. Keep them off for the light check.
# Turn them on when you want to rerun the paper metrics or fine-tuning.
RUN_MODEL_EVALUATION = False
RUN_SYNTHETIC_FINE_TUNING = False
SAVE_EXAMPLE_PREDICTIONS = False

# Turn this on to build the model from the code below.
# It needs ultralytics, torchvision, and opencv-python.
BUILD_MULTITASK_MODEL = False

IMAGE_SIZE = 640
BATCH_SIZE = 8
CONFIDENCE = 0.25
MATCH_IOU = 0.50
SUBSET_SEED = 42

PAPER_METRIC_TOLERANCE = 0.01


## 1. Environment and Paths

I keep generated files under `runs/perception_reproduction/`. That folder is ignored by Git, so rerunning the notebook will not change the source files.


In [ ]:
def run(command, *, cwd=None, check=True):
    """Run a shell command and show its output."""
    print('$', ' '.join(str(part) for part in command))
    result = subprocess.run(
        [str(part) for part in command],
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout)
    if check and result.returncode:
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {command}')
    return result


def find_repo_root(start: Path) -> Path:
    """Find the repo even if the notebook was opened from notebooks/."""
    for candidate in [start, *start.parents]:
        if (candidate / 'README.md').is_file() and (candidate / 'unity').is_dir():
            return candidate
    raise FileNotFoundError('Could not find the RDMO-DigitalTwin repository root.')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)

DATA_ROOT = Path(os.environ.get('RDMO_DATA_ROOT', REPO_ROOT / 'data')).expanduser().resolve()
MODELS_ROOT = REPO_ROOT / 'models'
OUTPUT_ROOT = Path(os.environ.get('RDMO_RUNS_DIR', REPO_ROOT / 'runs' / 'perception_reproduction')).expanduser().resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('Repository root:', REPO_ROOT)
print('Data root:', DATA_ROOT)
print('Output root:', OUTPUT_ROOT)


In [ ]:
if RUN_INSTALL_DEPS:
    # pandas/pyyaml are enough for dataset checks; the model cells need the rest.
    packages = [
        'pandas',
        'matplotlib',
        'pyyaml',
        'torch',
        'torchvision',
        'ultralytics==8.4.76',
        'opencv-python',
        'numpy',
        'tqdm',
    ]
    run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--prefer-binary', *packages])

import pandas as pd
try:
    import yaml
except ImportError as error:
    raise ImportError('Install pyyaml or set RUN_INSTALL_DEPS=True in the first cell.') from error


## 2. Paper Values

These are the numbers reported in the paper. Later cells recompute what they can from the local files and show the difference.


In [ ]:
EXPECTED_DATASETS = pd.DataFrame([
    {
        'folder': 'merged_dataset',
        'paper_name': 'Merged Dataset',
        'images': 18741,
        'annotated_images': 17938,
        'background_images': 803,
        'boxes': 71034,
    },
    {
        'folder': 'balanced_dataset',
        'paper_name': 'Balanced Dataset',
        'images': 46175,
        'annotated_images': 42755,
        'background_images': 3420,
        'boxes': 120769,
    },
    {
        'folder': 'synthetic_dataset',
        'paper_name': 'Synthetic Dataset',
        'images': 2235,
        'annotated_images': 2235,
        'background_images': 0,
        'boxes': 25943,
    },
])

EXPECTED_CLASS_COUNTS = pd.DataFrame([
    {'folder': 'merged_dataset', 'Single Crack': 27536, 'Crocodile Crack': 7796, 'Pothole': 1800, 'Person': 17034, 'Car': 16868},
    {'folder': 'balanced_dataset', 'Single Crack': 24295, 'Crocodile Crack': 24295, 'Pothole': 24295, 'Person': 24000, 'Car': 23884},
    {'folder': 'synthetic_dataset', 'Single Crack': 4435, 'Crocodile Crack': 3134, 'Pothole': 4665, 'Person': 8376, 'Car': 5333},
])

EXPECTED_MODEL_METRICS = pd.DataFrame([
    {'checkpoint': 'model_base.pt', 'metric': 'final_mAP50', 'paper': 0.7442},
    {'checkpoint': 'model_base.pt', 'metric': 'final_mAP50_95', 'paper': 0.6124},
    {'checkpoint': 'model_base.pt', 'metric': 'final_macro_F1', 'paper': 0.7666},
    {'checkpoint': 'model_finetuned.pt', 'metric': 'final_mAP50', 'paper': 0.9591},
    {'checkpoint': 'model_finetuned.pt', 'metric': 'final_mAP50_95', 'paper': 0.6450},
    {'checkpoint': 'model_finetuned.pt', 'metric': 'final_macro_F1', 'paper': 0.9400},
])

display(EXPECTED_DATASETS)
display(EXPECTED_CLASS_COUNTS)
display(EXPECTED_MODEL_METRICS)


## 3. Check the Dataset Files

Here I read the YOLO labels and count images, annotated images, background images, boxes, and class instances.

One detail worth keeping visible: my local `merged_dataset` is the YOLO-ready annotated copy, so it may not include the background-only images listed in the paper. The box and class counts should still match.


In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
DATASET_FOLDERS = ['merged_dataset', 'balanced_dataset', 'synthetic_dataset']


def load_dataset_yaml(dataset_root: Path) -> dict:
    yaml_path = dataset_root / 'dataset.yaml'
    if not yaml_path.is_file():
        yaml_path = dataset_root / 'data.yaml'
    if not yaml_path.is_file():
        raise FileNotFoundError(f'Missing dataset YAML in {dataset_root}')
    with yaml_path.open('r', encoding='utf-8') as stream:
        return yaml.safe_load(stream)


def split_image_dirs(dataset_root: Path) -> list[tuple[str, Path, Path]]:
    """Return the image and label folders for each split."""
    config = load_dataset_yaml(dataset_root)
    split_rows = []
    for logical_split in ['train', 'val', 'test']:
        image_rel = config.get(logical_split)
        if not image_rel:
            continue
        image_dir = (dataset_root / image_rel).resolve()
        label_dir = image_dir.parent / 'labels'
        split_name = image_dir.parent.name
        split_rows.append((split_name, image_dir, label_dir))
    return split_rows


def count_yolo_dataset(dataset_root: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Count images, labels, boxes, and class instances."""
    config = load_dataset_yaml(dataset_root)
    names = {int(key): value for key, value in config.get('names', {}).items()}
    split_rows = []
    class_counter = Counter()

    for split_name, image_dir, label_dir in split_image_dirs(dataset_root):
        if not image_dir.is_dir():
            raise FileNotFoundError(f'Missing image directory: {image_dir}')

        images = sorted(path for path in image_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)
        annotated_images = 0
        box_count = 0

        for image_path in images:
            label_path = label_dir / f'{image_path.stem}.txt'
            image_boxes = 0
            if label_path.is_file():
                with label_path.open('r', encoding='utf-8', errors='ignore') as stream:
                    for line in stream:
                        parts = line.strip().split()
                        if not parts:
                            continue
                        class_id = int(float(parts[0]))
                        class_counter[names.get(class_id, str(class_id))] += 1
                        image_boxes += 1
            annotated_images += int(image_boxes > 0)
            box_count += image_boxes

        split_rows.append({
            'split': split_name,
            'images': len(images),
            'annotated_images': annotated_images,
            'background_images': len(images) - annotated_images,
            'boxes': box_count,
        })

    split_df = pd.DataFrame(split_rows)
    class_df = pd.DataFrame([class_counter]).fillna(0).astype(int)
    return split_df, class_df


if RUN_DATASET_SCAN:
    dataset_total_rows = []
    dataset_class_rows = []

    for folder in DATASET_FOLDERS:
        dataset_root = DATA_ROOT / folder
        split_df, class_df = count_yolo_dataset(dataset_root)

        print(f'\n{folder}')
        display(split_df)

        totals = split_df[['images', 'annotated_images', 'background_images', 'boxes']].sum().to_dict()
        totals['folder'] = folder
        dataset_total_rows.append(totals)

        class_row = class_df.iloc[0].to_dict()
        class_row['folder'] = folder
        dataset_class_rows.append(class_row)

    actual_dataset_totals = pd.DataFrame(dataset_total_rows)
    actual_class_counts = pd.DataFrame(dataset_class_rows).fillna(0)
else:
    actual_dataset_totals = pd.DataFrame()
    actual_class_counts = pd.DataFrame()
    print('RUN_DATASET_SCAN=False; skipped dataset counting.')


In [ ]:
if RUN_DATASET_SCAN:
    dataset_comparison = EXPECTED_DATASETS.merge(
        actual_dataset_totals,
        on='folder',
        how='left',
        suffixes=('_paper', '_actual'),
    )
    for metric in ['images', 'annotated_images', 'background_images', 'boxes']:
        dataset_comparison[f'{metric}_delta'] = dataset_comparison[f'{metric}_actual'] - dataset_comparison[f'{metric}_paper']

    class_comparison = EXPECTED_CLASS_COUNTS.merge(
        actual_class_counts,
        on='folder',
        how='left',
        suffixes=('_paper', '_actual'),
    )
    for class_name in ['Single Crack', 'Crocodile Crack', 'Pothole', 'Person', 'Car']:
        class_comparison[f'{class_name}_delta'] = class_comparison[f'{class_name}_actual'] - class_comparison[f'{class_name}_paper']

    display(dataset_comparison)
    display(class_comparison)

    dataset_comparison.to_csv(OUTPUT_ROOT / 'dataset_total_comparison.csv', index=False)
    class_comparison.to_csv(OUTPUT_ROOT / 'dataset_class_comparison.csv', index=False)
    print('Saved dataset comparisons under:', OUTPUT_ROOT)


## 4. Checkpoints and Model Implementation

The detector is a multitask YOLOv8n model. The first head detects three coarse classes: `Road-defect-general`, `Person`, and `Car`. A second head receives ROI-aligned road-defect features and predicts the final defect subtype: `Crocodile Crack`, `Single Crack`, or `Pothole`.

I define the model here first, then load the released checkpoints to check what is inside.


In [ ]:
BASE_CHECKPOINT = MODELS_ROOT / 'model_base.pt'
FINETUNED_CHECKPOINT = MODELS_ROOT / 'model_finetuned.pt'
CHECKPOINTS = [BASE_CHECKPOINT, FINETUNED_CHECKPOINT]

for checkpoint in CHECKPOINTS:
    if not checkpoint.is_file():
        raise FileNotFoundError(f'Missing checkpoint: {checkpoint}')
    print(f'{checkpoint.name}: {checkpoint.stat().st_size / 1024**2:.1f} MB')


### Model Definition

This is the same structure used by the PyTorch inference server in Unity:

1. rebuild YOLOv8n with three coarse detection classes;
2. load the `yolo.*` checkpoint weights;
3. rebuild the subtype head from `subtype_head.*` weights;
4. use ROIAlign on a YOLO feature map for each road-defect box;
5. replace `Road-defect-general` with the predicted subtype.

By default, this cell only defines the classes. Set `BUILD_MULTITASK_MODEL=True` to instantiate the model from `models/model_finetuned.pt`.


In [ ]:
try:
    import torch
    import torch.nn as nn
except ImportError:
    torch = None
    nn = None
    print('PyTorch is not installed. Set RUN_INSTALL_DEPS=True before building the notebook model.')


if torch is not None:
    class SubtypeHead(nn.Module):
        """Small classifier for ROI-aligned road-defect features."""

        def __init__(self, in_channels: int, conv_channels: int, hidden_units: int, out_classes: int):
            super().__init__()
            self.network = nn.Sequential(
                nn.Conv2d(in_channels, conv_channels, kernel_size=3, padding=1),
                nn.BatchNorm2d(conv_channels),
                nn.SiLU(inplace=True),
                nn.AdaptiveAvgPool2d((1, 1)),
                nn.Flatten(),
                nn.Linear(conv_channels, hidden_units),
                nn.ReLU(inplace=True),
                nn.Dropout(0.0),
                nn.Linear(hidden_units, out_classes),
            )

        def forward(self, roi_features):
            return self.network(roi_features)


    def extract_prefixed_state_dict(state_dict: dict, prefix: str) -> dict:
        """Drop a checkpoint prefix, such as yolo. or subtype_head."""
        return {
            key[len(prefix):]: value
            for key, value in state_dict.items()
            if key.startswith(prefix)
        }


    def build_subtype_head_from_state(state: dict, device: str):
        """Rebuild the subtype head from the saved tensor shapes."""
        conv_weight = state.get('network.0.weight')
        hidden_weight = state.get('network.5.weight')
        output_weight = state.get('network.8.weight')
        if conv_weight is None or hidden_weight is None or output_weight is None:
            raise RuntimeError('The checkpoint does not contain the expected subtype_head tensors.')

        head = SubtypeHead(
            in_channels=int(conv_weight.shape[1]),
            conv_channels=int(conv_weight.shape[0]),
            hidden_units=int(hidden_weight.shape[0]),
            out_classes=int(output_weight.shape[0]),
        )
        missing, unexpected = head.load_state_dict(state, strict=False)
        if missing:
            print('Missing subtype-head keys:', missing)
        if unexpected:
            print('Unexpected subtype-head keys:', unexpected)
        return head.to(device).eval()


    class MultitaskYOLOv8n(nn.Module):
        """YOLOv8n detector with a second head for road-defect subtypes."""

        def __init__(self, yolo, subtype_head, *, image_size: int, roi_size: int, detection_names, subtype_names):
            super().__init__()
            self.yolo = yolo
            self.subtype_head = subtype_head
            self.image_size = int(image_size)
            self.roi_size = int(roi_size)
            self.detection_names = tuple(detection_names)
            self.subtype_names = tuple(subtype_names)

        @classmethod
        def from_checkpoint(cls, checkpoint_path: Path, device=None):
            """Load the detector and subtype head from one project checkpoint."""
            from ultralytics import YOLO
            from ultralytics.nn.tasks import DetectionModel, yaml_model_load

            device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
            checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

            detection_names = tuple(checkpoint.get('detection_names', ('Road-defect-general', 'Person', 'Car')))
            subtype_names = tuple(checkpoint.get('subtype_names', ('Crocodile Crack', 'Single Crack', 'Pothole')))
            image_size = int(checkpoint.get('image_size', IMAGE_SIZE))
            roi_size = int(checkpoint.get('roi_size', 7))

            yolo_state = extract_prefixed_state_dict(checkpoint['state_dict'], 'yolo.')
            subtype_state = extract_prefixed_state_dict(checkpoint['state_dict'], 'subtype_head.')

            cfg = yaml_model_load('yolov8n.yaml')
            detector_model = DetectionModel(cfg, ch=3, nc=len(detection_names), verbose=False).to(device)
            current_state = detector_model.state_dict()
            compatible_yolo_state = {
                key: value
                for key, value in yolo_state.items()
                if key in current_state and tuple(current_state[key].shape) == tuple(value.shape)
            }
            detector_model.load_state_dict(compatible_yolo_state, strict=False)

            yolo = YOLO('yolov8n.yaml')
            yolo.model = detector_model
            yolo.model.names = {index: name for index, name in enumerate(detection_names)}
            yolo.overrides['mode'] = 'predict'
            yolo.overrides['save'] = False
            yolo.overrides['verbose'] = False

            subtype_head = build_subtype_head_from_state(subtype_state, device)
            model = cls(
                yolo,
                subtype_head,
                image_size=image_size,
                roi_size=roi_size,
                detection_names=detection_names,
                subtype_names=subtype_names,
            ).to(device).eval()
            model.loaded_yolo_tensors = len(compatible_yolo_state)
            model.skipped_yolo_tensors = len(yolo_state) - len(compatible_yolo_state)
            return model

        def describe(self) -> dict:
            return {
                'image_size': self.image_size,
                'roi_size': self.roi_size,
                'detection_names': list(self.detection_names),
                'subtype_names': list(self.subtype_names),
                'loaded_yolo_tensors': getattr(self, 'loaded_yolo_tensors', None),
                'skipped_yolo_tensors': getattr(self, 'skipped_yolo_tensors', None),
            }

        def image_to_tensor(self, image_bgr):
            """Resize a BGR image and convert it to a normalized tensor."""
            import cv2
            resized = cv2.resize(image_bgr, (self.image_size, self.image_size), interpolation=cv2.INTER_LINEAR)
            rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
            tensor = torch.from_numpy(rgb).permute(2, 0, 1).unsqueeze(0)
            return tensor.to(next(self.parameters()).device).float() / 255.0

        def collect_subtype_feature_map(self, image_tensor):
            """Run YOLO layer by layer and keep the feature map the subtype head expects."""
            target_channels = int(self.subtype_head.network[0].weight.shape[1])
            layers = self.yolo.model.model
            outputs = []
            x = image_tensor
            candidates = []

            for layer_index, layer in enumerate(layers):
                source = getattr(layer, 'f', -1)
                if source != -1:
                    if isinstance(source, int):
                        layer_input = outputs[source]
                    else:
                        layer_input = [x if item == -1 else outputs[item] for item in source]
                else:
                    layer_input = x

                x = layer(layer_input)
                outputs.append(x)
                if torch.is_tensor(x) and x.dim() == 4 and int(x.shape[1]) == target_channels:
                    _, _, height, width = x.shape
                    candidates.append((layer_index, x, int(height), int(width)))

            if not candidates:
                raise RuntimeError(f'No YOLO feature map has the {target_channels} channels expected by the subtype head.')

            # Use the sharpest compatible feature map; if there is a tie, take the later layer.
            candidates.sort(key=lambda item: (item[2] * item[3], item[0]))
            return candidates[-1][1]

        @torch.no_grad()
        def classify_damage_rois(self, image_bgr, boxes_xyxy):
            """Classify road-defect boxes from ROI-aligned YOLO features."""
            from torchvision.ops import roi_align

            if not boxes_xyxy:
                return []

            image_height, image_width = image_bgr.shape[:2]
            image_tensor = self.image_to_tensor(image_bgr)
            feature_map = self.collect_subtype_feature_map(image_tensor)
            _, _, feature_height, feature_width = feature_map.shape

            rois = []
            for x1, y1, x2, y2 in boxes_xyxy:
                scaled = [
                    float(x1) * self.image_size / float(image_width),
                    float(y1) * self.image_size / float(image_height),
                    float(x2) * self.image_size / float(image_width),
                    float(y2) * self.image_size / float(image_height),
                ]
                scaled[0] = max(0.0, min(float(self.image_size - 1), scaled[0]))
                scaled[1] = max(0.0, min(float(self.image_size - 1), scaled[1]))
                scaled[2] = max(scaled[0] + 1.0, min(float(self.image_size), scaled[2]))
                scaled[3] = max(scaled[1] + 1.0, min(float(self.image_size), scaled[3]))
                rois.append([0.0, *scaled])

            rois_tensor = torch.tensor(rois, dtype=torch.float32, device=feature_map.device)
            spatial_scale_x = feature_width / float(self.image_size)
            spatial_scale_y = feature_height / float(self.image_size)
            spatial_scale = float((spatial_scale_x + spatial_scale_y) / 2.0)

            roi_features = roi_align(
                feature_map,
                rois_tensor,
                output_size=(self.roi_size, self.roi_size),
                spatial_scale=spatial_scale,
                aligned=True,
            )
            probabilities = torch.softmax(self.subtype_head(roi_features), dim=1)
            confidences, predictions = torch.max(probabilities, dim=1)

            return [
                {
                    'subtype': self.subtype_names[int(predictions[index].item())],
                    'subtype_confidence': float(confidences[index].item()),
                    'subtype_probabilities': {
                        self.subtype_names[class_index]: float(probabilities[index, class_index].item())
                        for class_index in range(probabilities.shape[1])
                    },
                }
                for index in range(len(boxes_xyxy))
            ]

        @torch.no_grad()
        def predict(self, image_bgr, *, confidence: float = 0.40, iou: float = 0.45):
            """Run detection first, then subtype classification for road defects."""
            result = self.yolo.predict(
                image_bgr,
                imgsz=self.image_size,
                conf=confidence,
                iou=iou,
                verbose=False,
            )[0]

            detections = []
            road_boxes = []
            road_indices = []

            for box in result.boxes:
                class_id = int(box.cls[0].item())
                coarse_name = self.detection_names[class_id]
                coords = [float(value) for value in box.xyxy[0].tolist()]
                detection = {
                    'coarse_class': coarse_name,
                    'class_name': coarse_name,
                    'detection_confidence': float(box.conf[0].item()),
                    'box_xyxy': coords,
                }
                if coarse_name == 'Road-defect-general':
                    road_indices.append(len(detections))
                    road_boxes.append(coords)
                detections.append(detection)

            for detection_index, subtype in zip(road_indices, self.classify_damage_rois(image_bgr, road_boxes)):
                detections[detection_index]['class_name'] = subtype['subtype']
                detections[detection_index]['subtype_confidence'] = subtype['subtype_confidence']
                detections[detection_index]['subtype_probabilities'] = subtype['subtype_probabilities']

            return detections
else:
    print('Skipping model class definitions because PyTorch is unavailable.')


In [ ]:
notebook_model = None

if BUILD_MULTITASK_MODEL:
    if 'MultitaskYOLOv8n' not in globals():
        raise RuntimeError('MultitaskYOLOv8n is not defined. Install PyTorch and rerun the implementation cell.')
    notebook_model = MultitaskYOLOv8n.from_checkpoint(FINETUNED_CHECKPOINT)
    print(json.dumps(notebook_model.describe(), indent=2))
else:
    print('BUILD_MULTITASK_MODEL=False; model code is loaded but no model was built.')


### Checkpoint Metadata

Here I check the saved class names, input size, ROI size, and training metadata in the two checkpoint files.


In [ ]:
try:
    import torch
except ImportError:
    torch = None
    print('PyTorch is not installed. Set RUN_INSTALL_DEPS=True or install torch to inspect checkpoint metadata.')


def checkpoint_summary(path: Path) -> dict:
    checkpoint = torch.load(path, map_location='cpu', weights_only=False)
    metadata = checkpoint.get('metadata', {})
    history = metadata.get('history', []) if isinstance(metadata, dict) else []
    return {
        'file': path.name,
        'format_version': checkpoint.get('format_version'),
        'detector_weights': checkpoint.get('detector_weights'),
        'image_size': checkpoint.get('image_size'),
        'roi_size': checkpoint.get('roi_size'),
        'lambda_subtype': checkpoint.get('lambda_subtype'),
        'detection_names': list(checkpoint.get('detection_names', [])),
        'subtype_names': list(checkpoint.get('subtype_names', [])),
        'selected_epoch': metadata.get('epoch') if isinstance(metadata, dict) else None,
        'selection_metric': metadata.get('selection_metric') if isinstance(metadata, dict) else None,
        'selection_value': metadata.get('selection_value') if isinstance(metadata, dict) else None,
        'history_records': len(history),
    }


if torch is not None:
    checkpoint_summaries = pd.DataFrame([checkpoint_summary(path) for path in CHECKPOINTS])
    display(checkpoint_summaries)
    checkpoint_summaries.to_json(OUTPUT_ROOT / 'checkpoint_metadata.json', orient='records', indent=2)


## 5. Model Evaluation

The metric code used for the paper lives in the experiment package. I keep this off for a quick local check. Turn on `RUN_MODEL_EVALUATION` when you want to recompute mAP and macro F1 on `data/synthetic_dataset/test`.


In [ ]:
EXPERIMENT_CODE_URL = 'https://github.com/EdwinTSalcedo/rdmo-simulator.git'
EXPERIMENT_CODE_DIR = Path(os.environ.get('RDMO_EXPERIMENT_CODE_DIR', REPO_ROOT.parent / 'rdmo-simulator')).expanduser().resolve()


def locate_or_fetch_experiment_code() -> Path:
    """Use a local experiment package if it exists; otherwise clone it."""
    candidates = [
        REPO_ROOT,
        EXPERIMENT_CODE_DIR,
        Path('/content/rdmo-simulator'),
    ]
    for candidate in candidates:
        if (candidate / 'experiments' / 'common' / 'model.py').is_file():
            return candidate

    run(['git', 'clone', '--depth', '1', EXPERIMENT_CODE_URL, EXPERIMENT_CODE_DIR])
    if not (EXPERIMENT_CODE_DIR / 'experiments' / 'common' / 'model.py').is_file():
        raise FileNotFoundError('The companion experiment package was not found after cloning.')
    return EXPERIMENT_CODE_DIR


if RUN_MODEL_EVALUATION or RUN_SYNTHETIC_FINE_TUNING:
    CODE_ROOT = locate_or_fetch_experiment_code()
    if str(CODE_ROOT) not in sys.path:
        sys.path.insert(0, str(CODE_ROOT))
    print('Experiment code root:', CODE_ROOT)

    from experiments.workflow import ExperimentConfig, MultiTaskExperiment
    from experiments.synthetic_evaluation import SyntheticEvaluationConfig
    from experiments.synthetic_finetune import SyntheticFineTuneConfig, fine_tune_on_synthetic_dataset
else:
    print('RUN_MODEL_EVALUATION=False; skipping model evaluation.')


In [ ]:
def compact_model_metrics(report: dict) -> dict:
    return {
        'final_mAP50': report['final_5_class']['map50'],
        'final_mAP50_95': report['final_5_class']['map50_95'],
        'final_macro_F1': report['final_5_class']['macro_f1'],
        'detection_mAP50': report['detection']['map50'],
        'detection_mAP50_95': report['detection']['map50_95'],
        'detection_macro_F1': report['detection']['macro_f1'],
        'subtype_gt_macro_F1': report['subtype_ground_truth_rois']['macro_f1'],
        'subtype_pred_macro_F1': report['subtype_predicted_rois']['macro_f1'],
    }


def evaluate_checkpoint(checkpoint: Path, run_name: str, *, include_predictions: bool = False):
    output_dir = OUTPUT_ROOT / run_name
    config = ExperimentConfig(
        dataset_root=DATA_ROOT / 'synthetic_dataset',
        detector_weights='yolov8n.pt',
        output_dir=output_dir,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        epochs=1,
        confidence=CONFIDENCE,
        match_iou=MATCH_IOU,
        dataset_fraction=1.0,
        subset_seed=SUBSET_SEED,
        log_every_validation_images=25,
        save_evaluation_predictions=include_predictions,
    )
    experiment = MultiTaskExperiment(config)
    metadata = experiment.model.load_checkpoint(checkpoint)
    report = experiment.evaluate(use_best_checkpoint=False, include_predictions=include_predictions)
    return report, metadata


model_reports = {}

if RUN_MODEL_EVALUATION:
    for checkpoint in CHECKPOINTS:
        report, metadata = evaluate_checkpoint(
            checkpoint,
            f'evaluate_{checkpoint.stem}_on_synthetic_test',
            include_predictions=SAVE_EXAMPLE_PREDICTIONS and checkpoint == FINETUNED_CHECKPOINT,
        )
        metrics = compact_model_metrics(report)
        model_reports[checkpoint.name] = metrics
        print(f'\n{checkpoint.name}')
        print(json.dumps(metrics, indent=2))
else:
    print('RUN_MODEL_EVALUATION=False; skipped checkpoint evaluation.')


In [ ]:
if model_reports:
    observed_rows = []
    for checkpoint, metrics in model_reports.items():
        for metric, value in metrics.items():
            observed_rows.append({'checkpoint': checkpoint, 'metric': metric, 'observed': value})

    model_comparison = EXPECTED_MODEL_METRICS.merge(
        pd.DataFrame(observed_rows),
        on=['checkpoint', 'metric'],
        how='left',
    )
    model_comparison['delta'] = model_comparison['observed'] - model_comparison['paper']
    model_comparison['within_tolerance'] = model_comparison['delta'].abs() <= PAPER_METRIC_TOLERANCE
    display(model_comparison)
    model_comparison.to_csv(OUTPUT_ROOT / 'model_metric_comparison.csv', index=False)
else:
    display(EXPECTED_MODEL_METRICS)
    print('No observed model metrics yet. Set RUN_MODEL_EVALUATION=True to fill this in.')


## 6. Synthetic Fine-Tuning

`model_base.pt` is the checkpoint after the first training stage on the Balanced Dataset. This cell reruns the second stage, fine-tuning on the Synthetic Dataset.

I keep it off by default because it is the slowest part of the notebook and is much better with a GPU.


In [ ]:
if RUN_SYNTHETIC_FINE_TUNING:
    base_config = SyntheticEvaluationConfig(
        dataset_archive=DATA_ROOT / 'synthetic_dataset.zip',
        experiments_root=OUTPUT_ROOT,
        repo_root=REPO_ROOT,
        dataset_cache_dir=DATA_ROOT,
        local_dataset_archive=DATA_ROOT / 'synthetic_dataset.zip',
        train_run_name='shared_backbone_multitask_v1',
        eval_run_name='synthetic_finetune_yolov8n_eval',
        detector_weights='yolov8n.pt',
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        epochs=15,
        learning_rate=1e-4,
        lambda_subtype=1.0,
        train_dataset_fraction=1.0,
        eval_dataset_fraction=1.0,
        subset_seed=SUBSET_SEED,
        confidence=CONFIDENCE,
        match_iou=MATCH_IOU,
        log_every_batches=5,
        log_every_validation_images=25,
        save_evaluation_predictions=False,
    )

    fine_tune_config = SyntheticFineTuneConfig(
        base=base_config,
        output_dir=OUTPUT_ROOT / 'synthetic_finetune_yolov8n',
        epochs=10,
        batch_size=BATCH_SIZE,
        learning_rate=2e-5,
        weight_decay=1e-4,
        lambda_subtype=1.0,
        train_dataset_fraction=1.0,
        validation_dataset_fraction=1.0,
        test_dataset_fraction=1.0,
        workers=2,
        early_stopping_patience=5,
        lr_decay_patience=3,
        lr_decay_factor=0.5,
        min_learning_rate=1e-7,
    )

    fine_tune_result = fine_tune_on_synthetic_dataset(
        fine_tune_config,
        dataset_root=DATA_ROOT / 'synthetic_dataset',
        checkpoint=BASE_CHECKPOINT,
        evaluate_after_training=True,
    )

    print('Fine-tuning output:', fine_tune_result['output_dir'])
    print('Best checkpoint:', fine_tune_result['best_checkpoint'])
    print(json.dumps(compact_model_metrics(fine_tune_result['report']), indent=2))
else:
    print('RUN_SYNTHETIC_FINE_TUNING=False; skipping training.')


## 7. Outputs

The default run writes:

```text
runs/perception_reproduction/
|-- dataset_total_comparison.csv
|-- dataset_class_comparison.csv
`-- checkpoint_metadata.json
```

Model evaluation adds folders under `runs/perception_reproduction/evaluate_*`. Synthetic fine-tuning writes to `runs/perception_reproduction/synthetic_finetune_yolov8n/`.
